# Build controlled mechanisms, then combine them

Seasonality is a repeating pattern. Drift in a random walk is an expected increment. **Concept drift is a change in the generating law**: it can affect a trend, seasonal amplitude or period, noise scale, jump intensity, or regime transition probabilities. A fixed seasonal cycle is not itself concept drift.

This chapter builds the scalar implementations of four controlled process families and dispatches multivariate cases to [chapter 15](15_joint_processes.ipynb). Start with an isolated mechanism, hold the random seed fixed, then add one factor. These are synthetic diagnostic fixtures; they do not establish stock-prediction superiority. [Workflow](README.md#develop-the-package-here)

In [1]:
#| default_exp processes

## The information boundary

The model receives history and a future index. Generator parameters, future change schedules, hidden states, and reference trajectories stay with the evaluator. Reference forecasts condition on the observed history: GARCH variance is reconstructed from observed returns, and regime probabilities are filtered from observed returns rather than copied from the simulated hidden path.

Gaussian references remain exact where the conditional law is Gaussian. Jumps, changing volatility, and hidden regimes use analytic moments with conditional simulation. Quantiles from simulation are numerical estimates. With regime-specific jumps, the observation likelihood uses a bounded Poisson mixture; its omitted probability mass is recorded, and the conditional sampler is explicitly approximate.

For `n_targets > 1`, the joint laws in [15 · Joint controlled processes](15_joint_processes.ipynb) supply correlated shocks, shared jumps, CCC-GARCH, and shared hidden regimes. The scalar implementation below preserves its existing seeded behaviour.

In [2]:
#| export
from __future__ import annotations

from copy import deepcopy

import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import poisson

from timegym.contracts import ForecastTask, Reference, SimulationReference
from timegym._internal import _real_array

PROCESS_KINDS = ('composite', 'merton_jump_diffusion', 'garch', 'markov_switching')

_DEFAULTS = {
    'composite': dict(intercept=0., slope=0., amplitude=0., period=12., phase=0.,
                      sigma=1., residual='white_noise', jump_rate=0., jump_mean=0.,
                      jump_sigma=0., change_point=None, changes={}),
    'merton_jump_diffusion': dict(mu=.05, sigma=.2, jump_rate=.5, jump_mean=-.1,
                                 jump_sigma=.15, dt=1/252, initial_price=100., output='price'),
    'garch': dict(mu=0., omega=.00001, alpha=.08, beta=.9,
                  initial_variance=None, initial_price=100., output='log_return'),
    'markov_switching': dict(transition=[[.97, .03], [.08, .92]], initial_probs=[.8, .2],
                            regime_mu=[.08, -.15], regime_sigma=[.15, .45],
                            regime_jump_rate=[0., 0.], regime_jump_mean=[0., 0.],
                            regime_jump_sigma=[0., 0.], dt=1/252,
                            initial_price=100., output='log_return'),
}
_CHANGE_FIELDS = {'intercept', 'slope', 'amplitude', 'period', 'phase', 'sigma'}
_TAIL_TOLERANCE = 1e-12

In [3]:
#| export
from timegym._internal import positive_int, nonnegative_int

In [4]:
#| export
def _scalar(name, value, *, minimum=None, positive=False):
    a = _real_array(value, name)
    if a.ndim != 0 or not np.isfinite(a):
        raise ValueError(f'{name} must be a finite real scalar')
    result = float(a)
    if positive and result <= 0 or minimum is not None and result < minimum:
        raise ValueError(f'{name} must be {"positive" if positive else f">= {minimum}"}')
    return result

In [5]:
#| export
def _finite(value, purpose):
    a = np.asarray(value, dtype=float)
    if not np.isfinite(a).all():
        raise ValueError(f'{purpose} overflowed; reduce horizon, rates, scales or time step')
    return a

In [6]:
#| export
def _exp(value, purpose):
    with np.errstate(over='ignore', under='ignore', invalid='ignore'):
        result = np.exp(value)
    _finite(result, purpose)
    if np.any(result == 0):
        raise ValueError(f'{purpose} underflowed; reduce horizon, rates, scales or time step')
    return result

In [7]:
#| export
def _log_cumulant(k, mu, sigma, rate, jump_mean, jump_sigma, dt):
    """Physical-measure Merton convention: E[S(t+dt)/S(t)] = exp(mu*dt)."""
    with np.errstate(over='ignore', invalid='ignore'):
        kappa = np.expm1(jump_mean + .5 * jump_sigma**2)
        value = (k * (mu - .5*sigma**2 - rate*kappa) + .5*k*k*sigma**2
                 + rate*np.expm1(k*jump_mean + .5*k*k*jump_sigma**2)) * dt
    return _finite(value, 'Log-return cumulant')

## A small, explicit parameter vocabulary

Reject unknown parameters, unsupported output units, invalid probabilities and nonstationary GARCH variance. The extended processes currently require one target. Existing Gaussian families retain their multivariate support.

In [8]:
#| export
def resolve_process(kind, parameters, context_length, horizon, n_targets, seed):
    """Validate a controlled scalar or joint process; return JSON-safe parameters.

    ``composite`` changes apply at absolute time ``t >= change_point``. Its
    deterministic path is always ``intercept + slope*t + amplitude*sin(...)``;
    changing slope or intercept need not preserve continuity. The future
    schedule belongs to the evaluator, never the model's ForecastTask.
    """
    if kind not in PROCESS_KINDS:
        raise ValueError(f'Unknown controlled process {kind!r}')
    positive_int(context_length, 'context_length')
    positive_int(horizon, 'horizon')
    nonnegative_int(seed, 'seed')
    if positive_int(n_targets, 'n_targets') > 1:
        from timegym.joint_processes import resolve_joint_process
        return resolve_joint_process(kind, parameters, context_length, horizon, n_targets, seed)
    if not isinstance(parameters, dict):
        raise TypeError('parameters must be a dictionary')
    unknown = set(parameters) - set(_DEFAULTS[kind])
    if unknown:
        raise ValueError(f'Unsupported parameters for {kind}: {sorted(unknown)}')
    p = {**_DEFAULTS[kind], **parameters}
    if kind == 'composite':
        if p['residual'] not in ('white_noise', 'random_walk'):
            raise ValueError('residual must be white_noise or random_walk')
        for key in ('intercept', 'slope', 'amplitude', 'period', 'phase', 'sigma',
                    'jump_rate', 'jump_mean', 'jump_sigma'):
            p[key] = _scalar(key, p[key], positive=key == 'period',
                             minimum=0 if key in ('sigma', 'jump_rate', 'jump_sigma') else None)
        if p['jump_rate'] and p['residual'] != 'random_walk':
            raise ValueError('Compound-Poisson jumps require residual=random_walk')
        if not isinstance(p['changes'], dict):
            raise ValueError('changes must be a dictionary')
        if set(p['changes']) - _CHANGE_FIELDS:
            raise ValueError(f'changes may only contain {sorted(_CHANGE_FIELDS)}')
        p['changes'] = {key: _scalar(key, value, positive=key == 'period',
                                     minimum=0 if key == 'sigma' else None)
                        for key, value in p['changes'].items()}
        if p['change_point'] is not None:
            p['change_point'] = nonnegative_int(p['change_point'], 'change_point')
            if not p['changes']:
                raise ValueError('change_point requires nonempty changes')
        elif p['changes']:
            raise ValueError('changes require change_point')
        if p['jump_rate'] > 1e6:
            raise ValueError('jump_rate must be <= 1e6 per step for stable simulation')
        return p
    p['initial_price'] = _scalar('initial_price', p['initial_price'], positive=True)
    outputs = ('log_return', 'log_price') if kind == 'garch' else ('log_return', 'log_price', 'price')
    if p['output'] not in outputs:
        raise ValueError(f'{kind} output must be one of {outputs}')
    if kind == 'garch':
        for key in ('mu', 'omega', 'alpha', 'beta'):
            p[key] = _scalar(key, p[key], positive=key == 'omega',
                             minimum=0 if key in ('alpha', 'beta') else None)
        if p['alpha'] + p['beta'] >= 1:
            raise ValueError('GARCH requires alpha + beta < 1 for finite unconditional variance')
        p['initial_variance'] = (_scalar('initial_variance', p['initial_variance'], positive=True)
                                 if p['initial_variance'] is not None
                                 else p['omega'] / (1-p['alpha']-p['beta']))
        p['initial_variance'] = _scalar('initial_variance', p['initial_variance'], positive=True)
        return p
    p['dt'] = _scalar('dt', p['dt'], positive=True)
    if kind == 'merton_jump_diffusion':
        for key in ('mu', 'sigma', 'jump_rate', 'jump_mean', 'jump_sigma'):
            p[key] = _scalar(key, p[key], minimum=0 if key in ('sigma', 'jump_rate', 'jump_sigma') else None)
        if p['jump_rate']*p['dt'] > 1e6:
            raise ValueError('jump_rate * dt must be <= 1e6 for stable simulation')
        for k in (1., 2.):
            _log_cumulant(k, p['mu'], p['sigma'], p['jump_rate'], p['jump_mean'], p['jump_sigma'], p['dt'])
        return p
    transition = _real_array(p['transition'], 'transition')
    if transition.ndim != 2 or transition.shape[0] != transition.shape[1] or not len(transition):
        raise ValueError('transition must be a nonempty square matrix')
    n = len(transition)
    if not np.isfinite(transition).all() or np.any(transition < 0) or not np.allclose(transition.sum(1), 1, atol=1e-12, rtol=0):
        raise ValueError('transition must contain nonnegative probabilities with rows summing to one')
    p['transition'] = (transition / transition.sum(1, keepdims=True)).tolist()
    for key in ('initial_probs', 'regime_mu', 'regime_sigma', 'regime_jump_rate', 'regime_jump_mean', 'regime_jump_sigma'):
        a = _real_array(p[key], key)
        if a.shape != (n,) or not np.isfinite(a).all():
            raise ValueError(f'{key} must have one finite entry per regime')
        if key in ('initial_probs', 'regime_sigma', 'regime_jump_rate', 'regime_jump_sigma') and np.any(a < 0):
            raise ValueError(f'{key} must be nonnegative')
        if key == 'initial_probs':
            if not np.isclose(a.sum(), 1, rtol=0, atol=1e-12):
                raise ValueError('initial_probs must sum to one')
            a = a/a.sum()
        p[key] = a.tolist()
    sigma = np.asarray(p['regime_sigma'])
    rate = np.asarray(p['regime_jump_rate'])
    jump_sigma = np.asarray(p['regime_jump_sigma'])
    has_atoms = np.any(sigma == 0)
    has_continuous = np.any((sigma > 0) | ((rate > 0) & (jump_sigma > 0)))
    if has_atoms and has_continuous:
        raise ValueError('Markov emissions must not mix point masses and continuous densities; use strictly positive regime_sigma in every regime, or entirely discrete emissions')
    if has_atoms and p['output'] != 'log_return' and context_length > 1 and not _common_deterministic_return(p):
        raise ValueError('Discrete Markov emissions require output=log_return when conditioning on returns; reconstructing discrete returns from price/log_price loses exact support. A length-one level context or identical deterministic regimes is supported')
    if np.max(np.asarray(p['regime_jump_rate'])*p['dt']) > 100:
        raise ValueError('regime_jump_rate * dt must be <= 100 for the bounded Poisson-mixture filter')
    for k in (1., 2.):
        _log_cumulant(k, *[np.asarray(p[key]) for key in ('regime_mu', 'regime_sigma', 'regime_jump_rate', 'regime_jump_mean', 'regime_jump_sigma')], p['dt'])
    return p

In [9]:
#| export
def process_label(kind, parameters=None, *, target_index=None):
    """Human-readable mechanism and output units, retaining stable kind IDs."""
    if kind not in PROCESS_KINDS:
        return kind
    p = {**_DEFAULTS[kind], **({} if parameters is None else parameters)}
    if target_index is not None and kind == 'composite':
        p = deepcopy(p)
        for key in _CHANGE_FIELDS:
            value = np.asarray(p[key])
            if value.ndim == 1:
                p[key] = value[target_index]
            if key in p.get('changes', {}):
                value = np.asarray(p['changes'][key])
                if value.ndim == 1:
                    p['changes'][key] = value[target_index]
    if kind == 'composite':
        pieces = []
        if np.any(np.asarray(p['slope']) != 0) or np.any(np.asarray(p.get('changes', {}).get('slope', 0)) != 0):
            pieces.append('Linear trend')
        if np.any(np.asarray(p['amplitude']) != 0) or np.any(np.asarray(p.get('changes', {}).get('amplitude', 0)) != 0):
            pieces.append('Seasonality')
        pieces.append('white noise' if p['residual'] == 'white_noise' else 'accumulated shocks')
        if p['jump_rate']:
            pieces.append('jumps')
        label = ' + '.join(pieces)
        return label[0].upper() + label[1:] + (' (parameter change)' if p['change_point'] is not None else '')
    labels = {'merton_jump_diffusion': 'Geometric Brownian motion' if p.get('jump_rate', 0.) == 0 else 'Merton jump diffusion',
              'garch': 'CCC-GARCH(1,1)' if np.asarray(p.get('mu', 0.)).size > 1 else 'GARCH(1,1)'}
    if kind == 'markov_switching':
        labels[kind] = 'Hidden regimes + jumps' if any(p['regime_jump_rate']) else 'Hidden Markov regimes'
    units = {'price': 'price', 'log_price': 'log price', 'log_return': 'log returns'}
    return f'{labels[kind]} ({units[p["output"]]})'

## Pair experiments without sharing future information

Independent named random streams separate diffusion, jump counts, jump sizes, and regime transitions. Changing jump settings preserves the Gaussian innovations at a fixed seed. Each reference call starts a fresh local stream in a separate evaluator namespace, so matching observation and reference seed integers never reuse innovations.

In [10]:
#| export
def _streams(seed):
    """Named independent streams preserve diffusion when jump settings change."""
    keys = ('diffusion', 'jump_counts', 'jump_sizes', 'states')
    return dict(zip(keys, map(np.random.default_rng, np.random.SeedSequence(seed).spawn(len(keys)))))

In [11]:
#| export
def _reference_streams(seed):
    """Separate evaluator draws from observation streams even at equal seeds."""
    keys = ('diffusion', 'jump_counts', 'jump_sizes', 'states')
    sequence = np.random.SeedSequence([0x5447594D, seed])
    return dict(zip(keys, map(np.random.default_rng, sequence.spawn(len(keys)))))

In [12]:
#| export
def _reference_rng(seed):
    return np.random.default_rng(np.random.SeedSequence([0x5447594D, seed]))

In [13]:
#| export
def _jump_increment(rng, size, rate, mean, sigma):
    count = rng['jump_counts'].poisson(rate, size=size)
    return count*mean + np.sqrt(count)*sigma*rng['jump_sizes'].standard_normal(size)

In [14]:
#| export
def _merton_increment(rng, size, mu, sigma, rate, jump_mean, jump_sigma, dt):
    with np.errstate(over='ignore', invalid='ignore'):
        drift = (mu-.5*sigma**2-rate*np.expm1(jump_mean+.5*jump_sigma**2))*dt
    return _finite(drift + sigma*np.sqrt(dt)*rng['diffusion'].standard_normal(size)
                   + _jump_increment(rng, size, rate*dt, jump_mean, jump_sigma), 'Simulated log return')

In [15]:
#| export
def _frames(values, length, horizon):
    names = ('y',)
    future_index = pd.RangeIndex(length, length+horizon)
    context = pd.DataFrame(_finite(values[:length], 'Generated context'), columns=names, index=pd.RangeIndex(length))
    observed = pd.DataFrame(_finite(values[length:], 'Generated future'), columns=names, index=future_index)
    return ForecastTask(context, future_index), observed

In [16]:
#| export
def _case(kind, p, length, horizon, seed, values, mean, covariance, conditioning, sampler=None, approximate=False):
    from timegym.scenarios import Case
    task, observed = _frames(values, length, horizon)
    mean = _finite(mean, 'Conditional mean').reshape(horizon, 1)
    covariance = _finite(covariance, 'Conditional covariance')
    covariance = (covariance+covariance.T)/2
    # Only roundoff on a theoretically nonnegative diagonal is removable.
    diagonal = np.diag(covariance).copy()
    tolerance = 1e-10*max(1., float(np.max(np.abs(covariance))))
    if diagonal.min() < -tolerance:
        raise ValueError('Conditional variance lost numerical precision; rescale the process')
    np.fill_diagonal(covariance, np.maximum(diagonal, 0))
    conditioning = dict(observed_through=length-1, known_generator_parameters=True,
                        future_observations_used=False, **conditioning)
    reference_args = dict(mean=mean, variance=np.diag(covariance).reshape(horizon, 1).copy(),
                          covariance=covariance, index=task.future_index.copy(),
                          target_names=('y',), conditioning=conditioning)
    if sampler is None:
        reference = Reference(**reference_args)
    else:
        reference = SimulationReference(**reference_args, sampler=sampler,
                                        sampler_kind='approximate_conditional' if approximate else 'exact_conditional')
    manifest = dict(schema_version=1, kind=kind, context_length=length, horizon=horizon,
                    n_targets=1, target_names=['y'], seed=seed, parameters=p,
                    reference_kind=reference.kind, conditioning=deepcopy(reference.conditioning), array_order='horizon_major_C')
    return Case(task, observed, reference, manifest)

## Trend, seasonality, and one residual mechanism

The deterministic path is $m_t=b+ct+A\sin(2\pi t/P+\phi)$. Add either fresh observation noise, $Y_t=m_t+\sigma_t Z_t$, or an accumulated residual, $Y_t=m_t+X_t$. The latter can add independent compound-Poisson increments to $X_t$.

A declared `change_point` changes only the supplied fields at absolute time $t\geq\tau$. The before/after intercept and slope use **absolute time**: changing either can create a discontinuity. This is intentional and explicit, not automatic continuity matching. The future schedule is evaluator-only. Generic additive jumps here are not Merton price dynamics.

In [17]:
#| export
def _composite(p, length, horizon, seed):
    t = np.arange(length+horizon, dtype=float)
    schedule = {}
    for key in _CHANGE_FIELDS:
        a = np.full(len(t), p[key], dtype=float)
        if p['change_point'] is not None and key in p['changes']:
            a[t >= p['change_point']] = p['changes'][key]
        schedule[key] = a
    with np.errstate(over='ignore', invalid='ignore'):
        signal = (schedule['intercept']+schedule['slope']*t + schedule['amplitude']
                  *np.sin(2*np.pi*t/schedule['period']+schedule['phase']))
    _finite(signal, 'Composite deterministic path')
    rng = _streams(seed)
    residual = np.empty(len(t))
    for i in range(len(t)):
        shock = schedule['sigma'][i]*rng['diffusion'].standard_normal()
        if p['residual'] == 'random_walk' and i == 0:
            residual[i] = 0.
        else:
            residual[i] = shock
            if p['residual'] == 'random_walk':
                residual[i] += residual[i-1] + _jump_increment(rng, None, p['jump_rate'], p['jump_mean'], p['jump_sigma'])
    values = signal+residual
    future_sigma = schedule['sigma'][length:]
    last_residual = float(values[length-1]-signal[length-1])
    if p['residual'] == 'white_noise':
        mean = signal[length:]
        covariance = np.diag(future_sigma**2)
    else:
        mean = signal[length:]+last_residual+np.arange(1, horizon+1)*p['jump_rate']*p['jump_mean']
        cumulative_variance = np.cumsum(future_sigma**2 + p['jump_rate']*(p['jump_sigma']**2+p['jump_mean']**2))
        covariance = cumulative_variance[np.minimum.outer(np.arange(horizon), np.arange(horizon))]
    conditioning = dict(residual=p['residual'], future_parameter_schedule_known=True,
                        schedule_convention='absolute_time_intercept_plus_slope_t',
                        residual_level_from_observed_history=last_residual if p['residual'] == 'random_walk' else None)
    sampler = None
    if p['jump_rate']:
        def sampler(n, sample_seed):
            rng = _reference_streams(sample_seed)
            out = np.empty((n, horizon, 1))
            previous = np.full(n, last_residual)
            for h in range(horizon):
                previous = previous+future_sigma[h]*rng['diffusion'].standard_normal(n)+_jump_increment(rng, n, p['jump_rate'], p['jump_mean'], p['jump_sigma'])
                out[:, h, 0] = signal[length+h]+previous
            return out
    return values, mean, covariance, conditioning, sampler, False

In [18]:
#| export
def _to_values(returns, p):
    if p['output'] == 'log_return':
        return returns
    log_values = np.r_[np.log(p['initial_price']), np.log(p['initial_price'])+np.cumsum(returns)]
    return _exp(log_values, 'Simulated price') if p['output'] == 'price' else log_values

In [19]:
#| export
def _observed_returns(values, p, length):
    history = np.asarray(values[:length])
    if p['output'] == 'log_return':
        return history
    return np.diff(np.log(history) if p['output'] == 'price' else history)

In [20]:
#| export
def _transform_samples(returns, p, last):
    if p['output'] == 'log_return':
        return returns[..., None]
    log_futures = (np.log(last) if p['output'] == 'price' else last)+np.cumsum(returns, axis=1)
    return (_exp(log_futures, 'Reference price draws') if p['output'] == 'price' else log_futures)[..., None]

## Merton jumps: distinguish returns, log levels, and prices

For one step, $R=(\mu-\sigma^2/2-\lambda\kappa)\Delta t+\sigma\sqrt{\Delta t}Z+\sum_{i=1}^N J_i$, with $N\sim\mathrm{Poisson}(\lambda\Delta t)$, normal log jumps, and $\kappa=\exp(m_J+s_J^2/2)-1$. Here $\mu$ is a **physical-measure expected proportional price growth rate**, not a risk-neutral pricing assumption. In particular, $E[S_{t+h}\mid S_t]=S_t\exp(\mu h\Delta t)$, not the exponential of expected log price. The zero-jump limit is geometric Brownian motion.

In [21]:
#| export
def _merton(p, length, horizon, seed):
    rng = _streams(seed)
    count = length+horizon-(p['output'] != 'log_return')
    args = [p[key] for key in ('mu', 'sigma', 'jump_rate', 'jump_mean', 'jump_sigma', 'dt')]
    returns = np.array([_merton_increment(rng, None, *args) for _ in range(count)])
    values = _to_values(returns, p)
    last = float(values[length-1])
    mu, sigma, rate, jm, js, dt = args
    a = (mu-.5*sigma**2-rate*np.expm1(jm+.5*js**2)+rate*jm)*dt
    v = (sigma**2+rate*(js**2+jm**2))*dt
    steps = np.arange(1, horizon+1)
    if p['output'] == 'log_return':
        mean, covariance = np.full(horizon, a), np.eye(horizon)*v
    elif p['output'] == 'log_price':
        mean, covariance = last+steps*a, np.minimum.outer(steps, steps)*v
    else:
        k1, k2 = [_log_cumulant(k, *args) for k in (1., 2.)]
        mean = _exp(np.log(last)+steps*k1, 'Conditional price mean')
        with np.errstate(over='ignore', invalid='ignore'):
            covariance = np.outer(mean, mean)*np.expm1(np.minimum.outer(steps, steps)*(k2-2*k1))
    def sampler(n, sample_seed):
        rng = _reference_streams(sample_seed)
        draws = np.column_stack([_merton_increment(rng, n, *args) for _ in range(horizon)])
        return _transform_samples(draws, p, last)
    conditioning = dict(output_units=p['output'], drift_convention='physical_expected_proportional_price_growth',
                        initialization='initial_price_at_index_zero' if p['output'] != 'log_return' else 'first_return_at_index_zero')
    gaussian = p['jump_rate'] == 0 and p['output'] != 'price'
    return values, mean, covariance, conditioning, None if gaussian else sampler, False

## GARCH: volatility clusters even when returns are uncorrelated

Gaussian GARCH(1,1) uses $h_{t+1}=\omega+\alpha(R_t-\mu)^2+\beta h_t$. Every simulated future trajectory updates variance using **its own innovations**. Multi-step expected variance is analytic, but the multi-step conditional distribution is not generally Gaussian.

The supported outputs are log returns and accumulated log price. Price output is deliberately rejected: exponentiating GARCH returns introduces moment-existence issues that a Gaussian approximation would conceal.

In [22]:
#| export
def _garch(p, length, horizon, seed):
    rng = np.random.default_rng(seed)
    count = length+horizon-(p['output'] != 'log_return')
    returns = np.empty(count)
    variance = p['initial_variance']
    for t in range(count):
        shock = np.sqrt(variance)*rng.standard_normal()
        returns[t] = p['mu']+shock
        variance = p['omega']+p['alpha']*shock**2+p['beta']*variance
    values = _to_values(returns, p)
    # Reconstruct the next variance from observed data only, including for log levels.
    next_variance = p['initial_variance']
    for value in _observed_returns(values, p, length):
        next_variance = p['omega']+p['alpha']*(value-p['mu'])**2+p['beta']*next_variance
    variances = np.empty(horizon)
    variances[0] = next_variance
    for h in range(1, horizon):
        variances[h] = p['omega']+(p['alpha']+p['beta'])*variances[h-1]
    last = float(values[length-1])
    if p['output'] == 'log_return':
        mean, covariance = np.full(horizon, p['mu']), np.diag(variances)
    else:
        mean = last+np.arange(1, horizon+1)*p['mu']
        covariance = np.cumsum(variances)[np.minimum.outer(np.arange(horizon), np.arange(horizon))]
    def sampler(n, sample_seed):
        rng = _reference_rng(sample_seed)
        variance = np.full(n, next_variance)
        draws = np.empty((n, horizon))
        for h in range(horizon):
            shock = np.sqrt(variance)*rng.standard_normal(n)
            draws[:, h] = p['mu']+shock
            variance = p['omega']+p['alpha']*shock**2+p['beta']*variance
        return _transform_samples(draws, p, last)
    conditioning = dict(output_units=p['output'], next_variance_from_observed_history=float(next_variance),
                        initialization='specified_initial_variance')
    return values, mean, covariance, conditioning, sampler, False

## Hidden regimes, optionally with regime-specific jumps

The finite Markov chain transitions **before each return**, including the first one. A forward filter updates the latent-state posterior using observed returns only. A forecast averages over that posterior and all future transitions, rather than revealing the realized regime path.

Mixed point-mass and continuous emissions are rejected: rounding a tiny continuous return onto a deterministic atom can change the posterior arbitrarily. Purely discrete regimes use exact support matching on log returns; price/log-price histories are supported only for identical deterministic regimes or a length-one context with no reconstructed returns.

The jump-mixture filter truncates each Poisson emission distribution at an omitted mass below $10^{-12}$. This is an emission-tail bound, **not a bound on posterior error**. Analytic moment formulas are conditional on that numerical posterior, and the reference records this limitation.

In [23]:
#| export
def _regime_arrays(p):
    return [np.asarray(p[key], dtype=float) for key in ('regime_mu', 'regime_sigma', 'regime_jump_rate', 'regime_jump_mean', 'regime_jump_sigma')]

In [24]:
#| export
def _common_deterministic_return(p):
    mu, sigma, rate, jm, js = _regime_arrays(p)
    return bool(np.all(sigma == 0) and np.all((rate == 0) | ((jm == 0) & (js == 0)))
                and np.all(mu == mu[0]))

In [25]:
#| export
def _emission_components(p):
    mu, sigma, rate, jm, js = _regime_arrays(p)
    dt = p['dt']
    components = []
    tails = []
    for m, s, r, j, z in zip(mu, sigma, rate, jm, js):
        intensity = r*dt
        limit = 0 if intensity == 0 else int(poisson.ppf(1-_TAIL_TOLERANCE, intensity))
        counts = np.arange(limit+1)
        logweights = poisson.logpmf(counts, intensity)
        centers = (m-.5*s*s-r*np.expm1(j+.5*z*z))*dt+counts*j
        variances = s*s*dt+counts*z*z
        components.append((logweights, centers, variances))
        tails.append(float(poisson.sf(limit, intensity)))
    return components, max(tails)

In [26]:
#| export
def _emission_loglikelihood(value, components, prior):
    # At an atom, probability mass dominates continuous density. This handles
    # deterministic regimes and deterministic jumps without arbitrary jitter.
    atom_terms = []
    density_terms = []
    for logweights, centers, variances in components:
        atom = (variances == 0) & (centers == value)
        atom_terms.append(logsumexp(logweights[atom]) if np.any(atom) else -np.inf)
        continuous = variances > 0
        if np.any(continuous):
            v = variances[continuous]
            density_terms.append(logsumexp(logweights[continuous]-.5*(np.log(2*np.pi*v)+(value-centers[continuous])**2/v)))
        else:
            density_terms.append(-np.inf)
    return np.asarray(atom_terms if np.any(np.isfinite(atom_terms) & (prior > 0)) else density_terms)

In [27]:
#| export
def _filter_regimes(returns, p):
    transition = np.asarray(p['transition'])
    posterior = np.asarray(p['initial_probs'])
    if _common_deterministic_return(p):
        # These regimes have identical deterministic emissions. Reconstructing
        # log returns from levels can introduce roundoff, but conveys no state
        # information. Propagating the prior is therefore exact.
        return posterior@np.linalg.matrix_power(transition, len(returns)), 0.
    components, tail = _emission_components(p)
    for value in returns:
        prior = posterior@transition
        with np.errstate(divide='ignore'):
            logweights = np.log(prior)+_emission_loglikelihood(value, components, prior)
        normalization = logsumexp(logweights)
        if not np.isfinite(normalization):
            raise ValueError('Observed return has zero numerical likelihood under the regime filter; rescale parameters or avoid degenerate indistinguishable emissions')
        posterior = np.exp(logweights-normalization)
    return posterior, tail

In [28]:
#| export
def _markov_price_moments(posterior, transition, mu, sigma, rate, jm, js, dt, last, horizon):
    """Stable law-of-total-covariance calculation, without raw-moment subtraction.

    Track price mean/variance conditional on each state. State-mixture
    variances use centered differences; multiplier variances use expm1.
    This preserves exact zero uncertainty for indistinguishable deterministic
    regimes and avoids cancellation of nearly equal large price moments.
    """
    growth = _exp(mu*dt, 'Regime expected price multiplier')
    with np.errstate(over='ignore', invalid='ignore'):
        squared_jump = np.exp(2*jm+js*js)*np.expm1(js*js) + np.expm1(jm+.5*js*js)**2
        spread = (sigma*sigma + rate*squared_jump)*dt
        growth_variance = growth*growth*np.expm1(spread)
        growth_second = growth*growth + growth_variance
    _finite(growth_second, 'Regime second price multiplier moment')
    probs = np.asarray(posterior, dtype=float)
    conditional_mean = np.full(len(probs), last)
    conditional_variance = np.zeros(len(probs))
    states = []
    mean = np.empty(horizon)
    for h in range(horizon):
        joint = probs[:, None]*transition
        next_probs = joint.sum(axis=0)
        supported = next_probs > 0
        weights = np.divide(joint, next_probs[None, :], out=np.zeros_like(joint), where=supported[None, :])
        # An anchor preserves identical conditional means exactly.
        anchor = conditional_mean[0]
        incoming_mean = anchor+(weights*(conditional_mean-anchor)[:, None]).sum(axis=0)
        incoming_variance = (weights*(conditional_variance[:, None]
                                      +(conditional_mean[:, None]-incoming_mean[None, :])**2)).sum(axis=0)
        conditional_variance = incoming_variance*growth_second + incoming_mean**2*growth_variance
        conditional_mean = incoming_mean*growth
        probs = next_probs/next_probs.sum()
        mean[h] = conditional_mean[0]+probs@(conditional_mean-conditional_mean[0])
        states.append((probs.copy(), conditional_mean.copy(), conditional_variance.copy()))
    covariance = np.empty((horizon, horizon))
    growth_transition = transition*growth
    for h, (probs, state_mean, state_variance) in enumerate(states):
        expected_growth = np.ones(len(probs))
        centered_mean = state_mean-mean[h]
        for j in range(h, horizon):
            if j > h:
                expected_growth = growth_transition@expected_growth
            covariance[h, j] = covariance[j, h] = probs@(expected_growth*state_variance
                + centered_mean*(state_mean*expected_growth-mean[j]))
    return mean, covariance

In [29]:
#| export
def _markov(p, length, horizon, seed):
    rng = _streams(seed)
    transition = np.asarray(p['transition'])
    arrays = _regime_arrays(p)
    count = length+horizon-(p['output'] != 'log_return')
    state = rng['states'].choice(len(transition), p=p['initial_probs'])
    returns = np.empty(count)
    for t in range(count):
        state = rng['states'].choice(len(transition), p=transition[state])
        returns[t] = _merton_increment(rng, None, *[a[state] for a in arrays], p['dt'])
    values = _to_values(returns, p)
    posterior, tail = _filter_regimes(_observed_returns(values, p, length), p)
    mu, sigma, rate, jm, js = arrays
    dt = p['dt']
    a = (mu-.5*sigma*sigma-rate*np.expm1(jm+.5*js*js)+rate*jm)*dt
    v = (sigma*sigma+rate*(js*js+jm*jm))*dt
    last = float(values[length-1])
    powers = [np.linalg.matrix_power(transition, h) for h in range(horizon+1)]
    if p['output'] == 'price':
        mean, covariance = _markov_price_moments(posterior, transition, *arrays, dt, last, horizon)
    else:
        probabilities = [posterior@powers[h] for h in range(1, horizon+1)]
        probabilities = [probs/probs.sum() for probs in probabilities]
        mean = np.array([a[0]+probs@(a-a[0]) for probs in probabilities])
        covariance = np.empty((horizon, horizon))
        for h, probs in enumerate(probabilities):
            centered = a-mean[h]
            covariance[h, h] = probs@(v+centered*centered)
            for j in range(h+1, horizon):
                future_mean = a[0]+powers[j-h]@(a-a[0])
                covariance[h, j] = covariance[j, h] = probs@(centered*(future_mean-mean[j]))
        if p['output'] == 'log_price':
            mean = last+np.cumsum(mean)
            covariance = covariance.cumsum(axis=0).cumsum(axis=1)
    def sampler(n, sample_seed):
        rng = _reference_streams(sample_seed)
        states = rng['states'].choice(len(transition), size=n, p=posterior)
        draws = np.empty((n, horizon))
        for h in range(horizon):
            uniforms = rng['states'].random(n)
            cumulative = transition[states].cumsum(axis=1)
            cumulative[:, -1] = 1.
            states = (uniforms[:, None] > cumulative).sum(axis=1)
            draws[:, h] = _merton_increment(rng, n, *[a[states] for a in arrays], dt)
        return _transform_samples(draws, p, last)
    has_jumps = bool(np.any(rate > 0))
    conditioning = dict(output_units=p['output'], state_posterior=posterior.tolist(),
                        state_inference='forward_filter_observed_returns_only',
                        hidden_states_exposed_to_model=False,
                        emission_poisson_tail_bound=tail,
                        filter_note='truncated_emission_mixture; tail_bound_is_not_a_posterior_error_bound' if has_jumps else 'Gaussian_emissions',
                        moments_conditioning='numerically_filtered_posterior' if has_jumps else 'filtered_posterior')
    return values, mean, covariance, conditioning, sampler, has_jumps

In [30]:
#| export
def generate_process(kind, parameters, context_length, horizon, n_targets=1, seed=0):
    """Generate held-out data and a history-conditioned evaluator reference.

    Finance-inspired families are controlled mechanisms, not claims of
    stock-prediction superiority. All random streams are local. Hidden states,
    future variance paths, jump times and scheduled changes never enter the
    model's task. Joint GARCH labels numerical cross-covariance estimates explicitly;
    regimes with jumps condition on a numerically filtered posterior.
    """
    p = resolve_process(kind, parameters, context_length, horizon, n_targets, seed)
    if n_targets > 1:
        from timegym.joint_processes import generate_joint_process
        return generate_joint_process(kind, p, context_length, horizon, n_targets, seed)
    builders = dict(composite=_composite, merton_jump_diffusion=_merton, garch=_garch, markov_switching=_markov)
    with np.errstate(over='ignore', invalid='ignore'):
        values, mean, covariance, conditioning, sampler, approximate = builders[kind](p, context_length, horizon, seed)
    return _case(kind, p, context_length, horizon, int(seed), values, mean, covariance, conditioning, sampler, approximate)

## One factor at a time

A fixed cycle does not change the noise law. The second case below changes its seasonal amplitude and observation noise at a declared time. It is a parameter-change stress test, not a claim that the model can know the future change in advance.

In [31]:
from timegym import Scenario

baseline = Scenario('composite', context_length=120, horizon=24,
    parameters={'amplitude': 2., 'period': 12., 'sigma': .3}, seed=11)
changed = Scenario('composite', context_length=120, horizon=24,
    parameters={'amplitude': 2., 'period': 12., 'sigma': .3,
                'change_point': 100, 'changes': {'amplitude': 4., 'sigma': .8}}, seed=11)
base_case, changed_case = baseline.generate(), changed.generate()
assert base_case.task.context.iloc[:100].equals(changed_case.task.context.iloc[:100])
assert changed_case.reference.conditioning['future_observations_used'] is False
print(baseline.display_name)
print(changed.display_name)

Seasonality + white noise
Seasonality + white noise (parameter change)
